In [ ]:
import sys
import os
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')
sys.path.append('/home/mvargas/code/Huawei/survan')

from lambda_cox import LambdaSA
from utils import get_churn_lastfm_dataset_months, get_churn_kkbox, get_targets_and_masks, train_test_split
import yaml
import jax
import jax.numpy as jnp
import haiku as hk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# path = '/Users/mariana/Documents/projects/Huawei/SurvanData/lastfm-dataset-1K'
path = '/home/mvargas/code/Huawei/SurvanData/kkbox-churn-prediction-challenge/kkbox'

In [ ]:
seqs, ts, cs = get_churn_kkbox(path)

In [ ]:
target, h_ws, mask = get_targets_and_masks(seqs, ts, cs, True)

In [5]:
X_train, X_test, y_train, y_test, hws_train, hws_test, \
        m_train, m_test, ts_train, ts_test, cs_train, cs_test = \
train_test_split(seqs, target, h_ws, mask, ts, cs, 32, test_size=0.2)

In [6]:
m_train[ts_train == 5][0][:6, :6]

array([[ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [False, False, False, False, False, False]])

In [8]:
df_events = pd.read_csv(os.path.join(path, 'events.csv'))

In [9]:
df_events.head()

,userid,month,censored,time
0,user_000001,2009-05,0,26
1,user_000002,2009-04,0,38
2,user_000003,2009-05,0,38
3,user_000004,2009-04,0,25
4,user_000005,2009-05,0,32


In [10]:
prof = pd.read_csv(os.path.join(path, 'userid-profile.tsv'), sep='\t')

In [11]:
len(prof)

992

In [21]:
prof.head()

,#id,gender,age,country,registered
0,user_000001,m,25.367133,Japan,"Aug 13, 2006"
1,user_000002,f,25.367133,Peru,"Feb 24, 2006"
2,user_000003,m,22.000000,United States,"Oct 30, 2005"
3,user_000004,f,25.367133,unknown,"Apr 26, 2006"
4,user_000005,m,25.367133,Bulgaria,"Jun 29, 2006"


In [13]:
prof['gender'] = prof.gender.fillna('unknown')

In [18]:
prof['country'] = prof.country.fillna('unknown')

In [20]:
average_age = prof['age'].mean()
prof['age'] = prof['age'].fillna(average_age)

In [27]:
aux = pd.get_dummies(prof, columns=['gender', 'country'])

In [28]:
bool_columns = aux.select_dtypes(include='bool').columns

# Convert boolean columns to float type
aux[bool_columns] = aux[bool_columns].astype(float)

In [2]:
#kkbox dataset
path = '/home/mvargas/code/Huawei/SurvanData/kkbox-churn-prediction-challenge/kkbox'

In [8]:
# Loading pre-processed kkbox data
path = '/home/mvargas/code/Huawei/SurvanData/kkbox-churn-prediction-challenge/kkbox'
cov = pd.read_feather(os.path.join(path, 'covariates_onehot.feather'))
logs = pd.read_feather(os.path.join(path, 'logs_preprocessed.feather'))
surv = pd.read_feather(os.path.join(path, 'survival_data.feather'))

In [31]:
surv.head()

,msno,churn,n_prev_churns,days_between_subs,start_date,registration_init_time,duration,days_since_reg_init,duration_censor,duration_lcd,churn_lcd,duration_censor_lcd,churn_type
0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,True,0,NaN,2016-09-09,2004-03-27,5,4549.0,142,5,True,142,final
1,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,True,0,NaN,2015-11-21,2012-12-24,410,1062.0,435,410,True,435,final
2,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,False,0,NaN,2016-11-16,2014-06-08,119,892.0,119,74,False,74,censoring
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,True,0,NaN,2015-01-31,2010-11-18,413,1535.0,729,413,True,729,30days
4,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,False,1,NaN,2016-07-31,2010-11-18,231,2082.0,231,182,False,182,censoring


In [30]:
logs.head()

,num_25,num_50,num_75,num_985,num_100,num_unq,log_minutes,msno
0,1.386294e+00,0.559616,-0.287682,0.223144,3.198673,3.208825,4.762232,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=
1,1.823216e-01,-0.510825,-0.762140,-0.916290,4.783595,4.713127,6.164910,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=
2,-2.014902e+00,-1.609437,-2.014902,-16.118096,4.753590,4.575398,6.154854,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=
3,1.000000e-07,-1.321755,-0.916290,-2.708049,4.740284,4.662810,6.154850,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=
4,8.472979e-01,-0.310155,-0.916290,-0.310155,4.304966,4.307662,5.744783,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=


In [45]:
months = surv.duration_lcd/30

In [46]:
months.max()

25.3

In [48]:
df['msno'] = logs.msno

In [65]:
sizes = filtered_df.groupby('msno').size()
sizes.max()

45

In [35]:
most_common_key = surv['msno'].mode().iloc[0]

# Extract columns corresponding to the most common key
cols_surv = surv[surv['msno'] == most_common_key].drop(columns='msno')

In [37]:
event = surv.groupby('msno')['churn'].all()

In [41]:
len(event)

2307537

In [66]:
len(sizes)

1368900

In [50]:
sizes.name = 'time'
event.name = 'event'

In [51]:
result = pd.merge(sizes, event, left_index=True, right_index=True, how='inner')

In [58]:
surv = result.reset_index()

In [61]:
len(surv)

1368900

In [62]:
keys_to_keep_df = pd.DataFrame({'Key': ['B', 'D', 'E']})
filtered_df = pd.merge(logs, surv, on='msno', how='left')

In [75]:
filtered_df = filtered_df.drop(columns=['time', 'event'])

In [77]:
filtered_df.to_feather(os.path.join(path, 'logs_filtered_preprocessed.feather'))

In [68]:
surv.to_feather(os.path.join(path, 'survival_preprocessed.feather'))

In [76]:
numeric_columns = filtered_df.select_dtypes(include=np.number).columns
numeric_columns

Index(['num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq',
       'log_minutes'],
      dtype='object')

In [78]:
filtered_df.columns

Index(['num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq',
       'log_minutes', 'msno'],
      dtype='object')